In [29]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.remote.webdriver import WebDriver
from selenium.webdriver.remote.webelement import WebElement

import logging
import time
from typing import Any, Dict, List, Optional
from datetime import datetime, timedelta
import time
import json
import pandas as pd

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)
logging.getLogger('selenium').setLevel(logging.INFO)

# Helpers

In [31]:
def create_driver(remote_url: str = "http://localhost:4444/wd/hub"):
    """Create a Selenium WebDriver connected to the Docker Selenium server."""
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1366,768")
    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = webdriver.Remote(
        command_executor=remote_url,
        options=chrome_options
    )
    
    # Execute CDP commands to hide automation
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
            Object.defineProperty(navigator, 'webdriver', {
                get: () => undefined
            })
        """
    })
    
    return driver

In [32]:
def wait_and_click_element_by_text(driver, text: str, timeout: int = 10) -> bool:
    """Wait for an element containing specific text and click it. Returns True if clicked, False if timeout."""
    try:
        element = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((By.XPATH, f"//*[contains(text(), '{text}')]"))
        )
        element.click()
        return True
    except TimeoutException:
        return False

In [33]:
def find_input_by_label_text(driver, label_text: str, input_text: str, timeout: int = 10) -> bool:
    """Find an input field near a label text and type into it. Returns True if successful, False if timeout."""
    try:
        # Find the label element by text
        label_element = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.XPATH, f"//*[contains(text(), '{label_text}')]"))
        )
        # Navigate to find the associated input (traverse DOM to find input in same container)
        parent = label_element.find_element(By.XPATH, "./ancestor::div[contains(@class, 'css-1dbjc4n')][1]")
        input_element = parent.find_element(By.CSS_SELECTOR, "input")
        input_element.click()
        input_element.clear()
        input_element.send_keys(input_text)
        return True
    except (TimeoutException, NoSuchElementException):
        return False

In [34]:
def get_element_by_data_testid(driver, data_testid: str, timeout: int = 10):
    """Wait for and return an element by data-testid attribute. Returns the element or None if timeout."""
    try:
        element = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, f"[data-testid='{data_testid}']"))
        )
        return element
    except TimeoutException:
        return None

# Init

In [35]:
driver = create_driver()
base_url = "https://www.traveloka.com/en-id/hotel"
driver.get(base_url)

# Search City, Hotel or Place to go

In [36]:
location = "Jakarta"
timeout = 10
search_bar_element = WebDriverWait(driver, timeout).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "input[placeholder='City, hotel, place to go']"))
)
search_bar_element.send_keys(location)

time.sleep(2)
item_name_element = get_element_by_data_testid(driver, "autocomplete-item-name")
item_name_element.click()

time.sleep(2)
submit_button_element = get_element_by_data_testid(driver, "search-submit-button")
submit_button_element.click()

# Parser

In [37]:
class HotelParser:
    """Parser for hotel information from Traveloka search results."""

    # CSS Selectors for hotel information
    _HOTEL_NAME = "[data-testid='tvat-hotelName']"
    _RATING_SCORE = "[data-testid='tvat-ratingScore']"
    _REVIEW_COUNT = "div[data-testid='tvat-ratingScore'] + div"
    _STAR_RATING = "[data-id='tvat-starRating']"
    _LOCATION = "[data-testid='tvat-hotelLocation']"
    _MAIN_IMAGE = "[data-testid='list-view-card-main-image']"
    _ORIGINAL_PRICE = "[data-testid='tvat-originalPrice']"
    _HOTEL_PRICE = "[data-testid='tvat-hotelPrice']"
    _TOTAL_PRICE = "[data-testid='charged-rooms-label']"
    _BOOKING_INFO = "[data-testid='fomo-text']"
    _HOTEL_TYPE_BADGE = "div[data-testid='tvat-hotelLocation'] img + div"
    _RANKING_BADGE = "div[dir='auto'][style*='background-image']"
    _FEATURE_BADGES = "[data-testid='hotel-feature-badge-0']"
    _SUPPORTING_IMAGES = "[data-testid='list-view-card-supporting-image']"

    def parse(self, hotel_container: WebElement) -> Optional[Dict[str, Any]]:
        """
        Parse hotel information from a hotel card container element.

        Args:
            hotel_container: Selenium WebElement containing the hotel card.

        Returns:
            Dictionary with hotel information or None if parsing fails.
        """
        try:
            logger.debug("Starting hotel parsing...")
            hotel_info: Dict[str, Any] = {}

            # Extract hotel name
            hotel_name_elem = self._safe_find_element(
                hotel_container, By.CSS_SELECTOR, self._HOTEL_NAME
            )
            hotel_info["hotel_name"] = (
                hotel_name_elem.text.strip() if hotel_name_elem else None
            )
            logger.debug(f"Hotel: {hotel_info.get('hotel_name')}")

            # Extract rating score
            rating_elem = self._safe_find_element(
                hotel_container, By.CSS_SELECTOR, self._RATING_SCORE
            )
            hotel_info["rating_score"] = (
                rating_elem.text.strip() if rating_elem else None
            )

            # Extract review count
            review_elem = self._safe_find_element(
                hotel_container, By.CSS_SELECTOR, self._REVIEW_COUNT
            )
            hotel_info["review_count"] = (
                self._extract_review_count(review_elem) if review_elem else None
            )

            # Extract star rating
            star_elem = self._safe_find_element(
                hotel_container, By.CSS_SELECTOR, self._STAR_RATING
            )
            hotel_info["star_rating"] = (
                self._extract_star_rating(star_elem) if star_elem else None
            )

            # Extract location
            location_elem = self._safe_find_element(
                hotel_container, By.CSS_SELECTOR, self._LOCATION
            )
            hotel_info["location"] = (
                location_elem.text.strip() if location_elem else None
            )

            # Extract main image URL
            main_image_elem = self._safe_find_element(
                hotel_container, By.CSS_SELECTOR, self._MAIN_IMAGE
            )
            hotel_info["main_image_url"] = (
                main_image_elem.get_attribute("src") if main_image_elem else None
            )

            # Extract supporting images URLs
            hotel_info["supporting_images"] = self._extract_supporting_images(
                hotel_container
            )

            # Extract original price
            original_price_elem = self._safe_find_element(
                hotel_container, By.CSS_SELECTOR, self._ORIGINAL_PRICE
            )
            hotel_info["original_price"] = (
                original_price_elem.text.strip() if original_price_elem else None
            )

            # Extract current price
            price_elem = self._safe_find_element(
                hotel_container, By.CSS_SELECTOR, self._HOTEL_PRICE
            )
            hotel_info["price"] = price_elem.text.strip() if price_elem else None

            # Extract total price (with taxes)
            total_price_elem = self._safe_find_element(
                hotel_container, By.CSS_SELECTOR, self._TOTAL_PRICE
            )
            hotel_info["total_price"] = (
                total_price_elem.text.strip() if total_price_elem else None
            )

            # Extract booking info (e.g., "Booked 8 times")
            booking_info_elem = self._safe_find_element(
                hotel_container, By.CSS_SELECTOR, self._BOOKING_INFO
            )
            hotel_info["booking_info"] = (
                booking_info_elem.text.strip() if booking_info_elem else None
            )

            # Extract hotel type (e.g., "Hotels")
            hotel_type_elem = self._safe_find_element(
                hotel_container, By.CSS_SELECTOR, self._HOTEL_TYPE_BADGE
            )
            hotel_info["hotel_type"] = (
                hotel_type_elem.text.strip() if hotel_type_elem else None
            )

            # Extract ranking badge (e.g., "No. 5 in Hotel with City View")
            ranking_elem = self._safe_find_element(
                hotel_container, By.CSS_SELECTOR, self._RANKING_BADGE
            )
            hotel_info["ranking"] = ranking_elem.text.strip() if ranking_elem else None

            # Extract feature badges (e.g., "Express check-out")
            hotel_info["features"] = self._extract_features(hotel_container)

            logger.debug(
                f"Parsed hotel: {hotel_info.get('hotel_name')} - {hotel_info.get('price')}"
            )
            return hotel_info

        except Exception as e:
            logger.error(f"Error parsing hotel info: {e}")
            return None

    def _safe_find_element(
        self, container: WebElement, by: By, selector: str
    ) -> Optional[WebElement]:
        """Safely find an element, returning None if not found."""
        try:
            return container.find_element(by, selector)
        except NoSuchElementException:
            return None

    def _extract_review_count(self, element: WebElement) -> Optional[str]:
        """Extract review count from element text (e.g., '(3,4K reviews)')."""
        if element:
            text = element.text.strip()
            return text.strip("()") if text else None
        return None

    def _extract_star_rating(self, element: WebElement) -> Optional[int]:
        """Extract star rating from data-rating attribute."""
        if element:
            rating = element.get_attribute("data-rating")
            try:
                return int(rating) if rating else None
            except ValueError:
                return None
        return None

    def _extract_supporting_images(
        self, hotel_container: WebElement
    ) -> List[str]:
        """Extract URLs of supporting images."""
        try:
            image_elems = hotel_container.find_elements(
                By.CSS_SELECTOR, self._SUPPORTING_IMAGES
            )
            return [
                img.get_attribute("src")
                for img in image_elems
                if img.get_attribute("src")
            ]
        except Exception as e:
            logger.debug(f"Error extracting supporting images: {e}")
            return []

    def _extract_features(self, hotel_container: WebElement) -> List[str]:
        """Extract hotel feature badges."""
        try:
            # Find all feature badge elements
            badge_elems = hotel_container.find_elements(
                By.CSS_SELECTOR, "[data-testid^='hotel-feature-badge-']"
            )
            features = []
            for badge in badge_elems:
                text = badge.text.strip()
                if text:
                    features.append(text)
            return features
        except Exception as e:
            logger.debug(f"Error extracting features: {e}")
            return []

    def parse_all(
        self, hotel_containers: List[WebElement], show_progress: bool = True
    ) -> List[Dict[str, Any]]:
        """
        Parse information from multiple hotel containers.

        Args:
            hotel_containers: List of WebElement hotel containers.
            show_progress: Whether to show progress information.

        Returns:
            List of dictionaries with hotel information.
        """
        total = len(hotel_containers)
        logger.info(f"Parsing data from {total} hotel containers...")

        hotels = []
        for i, container in enumerate(hotel_containers, 1):
            hotel_info = self.parse(container)
            if hotel_info:
                hotels.append(hotel_info)

            if show_progress and total > 0:
                percent = (i / total) * 100 if total > 0 else 0
                print(
                    f"\rParsing hotels: {i}/{total} ({percent:.1f}%)",
                    end="",
                    flush=True,
                )

        if show_progress and total > 0:
            print()  # New line after completion

        logger.info(f"Successfully parsed {len(hotels)}/{total} hotels")
        return hotels


def scroll_and_load_all_hotels(
    driver: WebDriver,
    scroll_pause: float = 2.0,
    num_scrolls: int = 5,
    num_hotels: int = 20,
) -> List[Dict[str, Any]]:
    """
    Scroll through the hotel list using window scroll and parse results incrementally.

    Parses current hotels before each scroll to avoid losing data when elements
    are recycled during infinite scroll.

    Args:
        driver: Selenium WebDriver instance.
        timeout: Maximum time in seconds to scroll.
        scroll_pause: Time to wait between scrolls in seconds.
        num_scrolls: Number of times to scroll (default: 5).

    Returns:
        list: List of parsed hotel information dictionaries.
    """
    all_hotels: List[Dict[str, Any]] = []
    seen_hotel_keys: set = set()
    last_count = 0
    no_change_count = 0
    max_no_change = 3  # Stop if no new hotels loaded after this many scrolls
    start_time = time.time()
    parser = HotelParser()

    # Parse initial hotels before any scrolling
    logger.info("Parsing initial hotels...")
    initial_containers = driver.find_elements(
        By.CSS_SELECTOR, "[data-testid='tvat-searchListItem']"
    )
    initial_hotels = parser.parse_all(initial_containers, show_progress=False)
    
    for hotel in initial_hotels:
        hotel_key = hotel.get("hotel_name")
        if hotel_key and hotel_key not in seen_hotel_keys:
            seen_hotel_keys.add(hotel_key)
            all_hotels.append(hotel)
    
    logger.info(f"Initial load: {len(initial_hotels)} hotels parsed, {len(all_hotels)} unique")
    print(f"Initial load: {len(all_hotels)} hotels")

    for i in range(num_scrolls):
        # Check if timeout reached
        if len(all_hotels) > num_hotels:
            logger.info(f"Threshold {num_hotels} number of hotels reached")
            break

        # Scroll to bottom of the window
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        # Wait for new content to load
        time.sleep(scroll_pause)

        # Wait for network to be idle (helps with lazy-loaded content)
        try:
            driver.execute_script("""
                return new Promise((resolve) => {
                    let pending = 0;
                    const origFetch = window.fetch;
                    window.fetch = function(...args) {
                        pending++;
                        return origFetch.apply(this, args).finally(() => {
                            pending--;
                            if (pending === 0) resolve();
                        });
                    };
                    if (pending === 0) resolve();
                    setTimeout(() => resolve(), 1000);
                });
            """)
        except Exception:
            pass  # Continue even if network wait fails

        # Count current hotels
        current_count = len(
            driver.find_elements(By.CSS_SELECTOR, "[data-testid='tvat-searchListItem']")
        )

        # Check if no new hotels loaded
        if current_count == last_count:
            no_change_count += 1
            if no_change_count >= max_no_change:
                logger.info(f"No new hotels loaded after {no_change_count} attempts")
                break
        else:
            no_change_count = 0
            logger.info(f"Scroll {i+1}/{num_scrolls}: {current_count} hotels visible")
            print(f"Scroll {i+1}/{num_scrolls}: {current_count} hotels visible")

            # Parse current visible hotels
            current_containers = driver.find_elements(
                By.CSS_SELECTOR, "[data-testid='tvat-searchListItem']"
            )
            current_hotels = parser.parse_all(current_containers, show_progress=False)
            
            # Add only new unique hotels
            new_count = 0
            for hotel in current_hotels:
                hotel_key = hotel.get("hotel_name")
                if hotel_key and hotel_key not in seen_hotel_keys:
                    seen_hotel_keys.add(hotel_key)
                    all_hotels.append(hotel)
                    new_count += 1
            
            if new_count > 0:
                logger.info(f"Added {new_count} new hotels, total: {len(all_hotels)}")
                print(f"Added {new_count} new hotels, total: {len(all_hotels)}")

        last_count = current_count

    logger.info(f"Scrolling complete. Total unique hotels parsed: {len(all_hotels)}")
    return all_hotels


def get_all_hotels(
    driver: WebDriver,
    scroll: bool = True,
    scroll_timeout: int = 60,
    num_scrolls: int = 20,
    num_hotels: int = 100,
) -> List[Dict[str, Any]]:
    """
    Get all hotels from the search results page with scroll and parse in one flow.

    This function follows the pattern:
    1. Wait for hotels to load
    2. Scroll to load more hotels (if enabled) - parses incrementally during scroll
    3. Returns all parsed hotels

    Args:
        driver: Selenium WebDriver instance.
        timeout: Maximum time to wait for hotels to load initially.
        scroll: Whether to scroll to load more hotels.
        scroll_timeout: Maximum time in seconds to scroll for more hotels.
        num_scrolls: Number of times to scroll (default: 5).

    Returns:
        list: List of hotel information dictionaries.
    """
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait

    try:
        # Wait for hotel cards to load
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "[data-testid='tvat-searchListItem']")
            )
        )

        # Scroll to load more hotels if enabled (this also parses incrementally)
        if scroll:
            logger.info("Scrolling to load all hotels...")
            hotels = scroll_and_load_all_hotels(
                driver, num_scrolls=num_scrolls, num_hotels=num_hotels
            )
            return hotels

        # If no scrolling, just parse current visible hotels
        hotel_containers = driver.find_elements(
            By.CSS_SELECTOR, "[data-testid='tvat-searchListItem']"
        )

        logger.info(f"Found {len(hotel_containers)} hotel containers")

        # Parse hotels
        parser = HotelParser()
        hotels = parser.parse_all(hotel_containers)

        return hotels

    except Exception as e:
        logger.error(f"Error getting hotels: {e}")
        return []


# Parse Hotel List

In [38]:
hotels = get_all_hotels(driver)

INFO:__main__:Scrolling to load all hotels...
INFO:__main__:Parsing initial hotels...
INFO:__main__:Parsing data from 16 hotel containers...
INFO:__main__:Successfully parsed 16/16 hotels
INFO:__main__:Initial load: 16 hotels parsed, 15 unique


Initial load: 15 hotels


INFO:__main__:Scroll 1/20: 12 hotels visible
INFO:__main__:Parsing data from 12 hotel containers...


Scroll 1/20: 12 hotels visible


INFO:__main__:Successfully parsed 12/12 hotels
INFO:__main__:Added 12 new hotels, total: 27


...

Added 12 new hotels, total: 96


INFO:__main__:Scroll 9/20: 28 hotels visible
INFO:__main__:Parsing data from 28 hotel containers...


Scroll 9/20: 28 hotels visible


INFO:__main__:Successfully parsed 28/28 hotels
INFO:__main__:Added 25 new hotels, total: 121
INFO:__main__:Threshold 100 number of hotels reached
INFO:__main__:Scrolling complete. Total unique hotels parsed: 121


Added 25 new hotels, total: 121


In [39]:
len(hotels)

121

# Save to CSV

In [40]:
df_hotels = pd.DataFrame(hotels)
df_hotels.to_csv(f"trave_hotels_{location.lower()}_{datetime.now().strftime('%Y%m%d%H%M%S')}.csv")

In [41]:
df_hotels

,hotel_name,rating_score,review_count,star_rating,location,main_image_url,supporting_images,original_price,price,total_price,booking_info,hotel_type,ranking,features
0,Mercure Jakarta Gatot Subroto,8.8/10,Very Good,4.0,"Gatot Subroto, South Jakarta",https://ik.imagekit.io/tvlk/apr-asset/rvN7CENf...,[https://ik.imagekit.io/tvlk/apr-asset/rvN7CEN...,Rp 1.320.000,Rp 990.000,Total Rp 1.197.900 \nfor 1 room,,None,None,[Express check-out]
1,Oakwood PIK Jakarta,8.3/10,Very Good,5.0,"Kamal Muara, Penjaringan",https://ik.imagekit.io/tvlk/apr-asset/dgXfoyh2...,[],Rp 1.801.801,Rp 1.351.351,,Booked 8 times,None,None,"[Cycling, Children play area, Babysitting, Mas..."
2,Somerset Berlian Jakarta,8.5/10,Very Good,4.0,"Permata Hijau, Kebayoran Lama",None,[],None,Rp 1.305.000,,Only 1 room(s) left at this price!,None,None,"[Great value in its class, Kitchenette, Fitnes..."
3,Model J Hotel Jakarta Soekarno - Hatta Airport,9.4/10,Exceptional,4.0,"Benda, Cengkareng",None,[],Rp 682.501,Rp 511.876,Total Rp 625.000 for 1 room,None,None,None,"[Clothes dryer, Fitness center, Express check-..."
4,Ashley Tanah Abang,9.0/10,Exceptional,4.0,"Tanah Abang, Central Jakarta",None,[],Rp 931.921,Rp 698.941,Total Rp 845.719 for 1 room,Booked 21 times,None,None,"[Located next to Downtown Area, Indoor heated ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,Golden Boutique Hotel Kemayoran,8.5,None,4.0,"Gunung Sahari, Kemayoran",None,[],None,Rp 448.767,Total Rp 547.945 for 1 room,None,None,None,[]
117,The Gunawarman,8.8,None,5.0,"Selong, Kebayoran Baru",None,[],None,Rp 1.834.562,Total Rp 2.240.000 for 1 room,None,None,None,[]
118,Amaris Hotel Satrio Kuningan - Jakarta,8.3/10,Very Good,2.0,"Kuningan, Setiabudi",None,[],None,Rp 517.690,,Booked 8 times,None,None,[]
119,Stanley Wahid Hasyim Jakarta,8.8/10,Very Good,NaN,"Kebon Sirih, Menteng",None,[],Rp 640.362,Rp 480.272,,Only 1 room(s) left at this price!,None,None,[Top picked by Budget Traveler]


# Exit Driver

In [28]:
driver.quit()